# 02 — Information Extraction

This notebook analyzes the information extraction part of the project.

The project extracts structured knowledge from scientific titles and abstracts. These outputs are later used to build a knowledge graph.

## Goals

- Load processed scientific documents.
- Extract named entities.
- Extract TF-IDF keywords.
- Extract simple co-occurrence relations.
- Save extraction outputs for inspection and graph construction.

In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()

for candidate in [current, *current.parents]:
    if (candidate / "src").exists() and (candidate / "config.yaml").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find project root")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.utils.common import (
    display_basic_frame_info,
    load_or_create_processed_documents,
    save_table,
    setup_notebook,
)

CONFIG, PATHS = setup_notebook()

## Load processed documents

In [ ]:
df = load_or_create_processed_documents(CONFIG, PATHS)

display_basic_frame_info(df, "Processed documents")

## Prepare text corpus

In [ ]:
texts = df["text"].fillna("").astype(str).tolist()
doc_ids = df["doc_id"].astype(str).tolist()

print("Number of documents:", len(texts))
print("First document preview:")
print(texts[0][:1000])

## Configure extraction modules

In [ ]:
extraction_cfg = CONFIG.get("extraction", {})

spacy_model = extraction_cfg.get("spacy_model", "en_core_web_sm")
entity_types = extraction_cfg.get("entity_types")
keyword_top_n = extraction_cfg.get("keyword_top_n", 10)
relation_window = extraction_cfg.get("relation_window", 1)

print("spaCy model:", spacy_model)
print("Entity types:", entity_types)
print("Keyword top_n:", keyword_top_n)
print("Relation window:", relation_window)

## Named Entity Recognition

In [ ]:
from src.extraction.entity_extraction import EntityExtractor

entity_extractor = EntityExtractor(
    spacy_model=spacy_model,
    entity_types=entity_types,
)

doc_entities = entity_extractor.extract_batch(texts)

entity_rows = []

for doc_id, entities in zip(doc_ids, doc_entities):
    for entity in entities:
        entity_rows.append(
            {
                "doc_id": doc_id,
                "entity": entity.text,
                "label": entity.label,
                "start": entity.start,
                "end": entity.end,
            }
        )

entities_df = pd.DataFrame(entity_rows)

display(entities_df.head(20))
print("Extracted entity rows:", len(entities_df))

## Entity type distribution

In [ ]:
if entities_df.empty:
    print("No entities extracted.")
else:
    entity_counts = entities_df["label"].value_counts().reset_index()
    entity_counts.columns = ["label", "count"]

    display(entity_counts)

    plt.figure(figsize=(8, 4))
    plt.bar(entity_counts["label"], entity_counts["count"])
    plt.title("Named Entity Type Distribution")
    plt.xlabel("Entity Type")
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## TF-IDF keyword extraction

In [ ]:
from src.extraction.keyword_extraction import TfidfKeywordExtractor

cleaned_texts = df["cleaned_text"].fillna("").astype(str).tolist()

keyword_extractor = TfidfKeywordExtractor(top_n=keyword_top_n)
keyword_extractor.fit(cleaned_texts)

keyword_rows = []

for index, doc_id in enumerate(doc_ids):
    for keyword, score in keyword_extractor.get_keywords(index):
        keyword_rows.append(
            {
                "doc_id": doc_id,
                "keyword": keyword,
                "score": float(score),
            }
        )

keywords_df = pd.DataFrame(keyword_rows)

display(keywords_df.head(20))
print("Extracted keyword rows:", len(keywords_df))

## Most frequent keywords

In [ ]:
if keywords_df.empty:
    print("No keywords extracted.")
else:
    top_keywords = (
        keywords_df.groupby("keyword", as_index=False)
        .agg(frequency=("doc_id", "count"), avg_score=("score", "mean"))
        .sort_values(["frequency", "avg_score"], ascending=False)
        .head(25)
    )

    display(top_keywords)

    plot_data = top_keywords.sort_values("frequency")

    plt.figure(figsize=(9, 6))
    plt.barh(plot_data["keyword"], plot_data["frequency"])
    plt.title("Most Frequent Extracted Keywords")
    plt.xlabel("Document Frequency")
    plt.ylabel("Keyword")
    plt.tight_layout()
    plt.show()

## Co-occurrence relation extraction

In [ ]:
from src.extraction.relation_extraction import (
    CooccurrenceRelationExtractor,
    aggregate_relations,
)

relation_extractor = CooccurrenceRelationExtractor(
    spacy_model=spacy_model,
    window=relation_window,
)

all_relations = []
relation_rows = []

for doc_id, text, entities in zip(doc_ids, texts, doc_entities):
    relations = relation_extractor.extract(text, entities)
    all_relations.extend(relations)

    for relation in relations:
        relation_rows.append(
            {
                "doc_id": doc_id,
                "source": relation.source,
                "relation": relation.relation,
                "target": relation.target,
                "weight": relation.weight,
            }
        )

relations_df = pd.DataFrame(relation_rows)

display(relations_df.head(20))
print("Extracted relation rows:", len(relations_df))

## Aggregated relations

In [ ]:
aggregated_relations = aggregate_relations(all_relations)

aggregated_df = pd.DataFrame(
    [
        {
            "source": relation.source,
            "relation": relation.relation,
            "target": relation.target,
            "weight": relation.weight,
        }
        for relation in aggregated_relations
    ]
)

if aggregated_df.empty:
    print("No aggregated relations available.")
else:
    display(aggregated_df.sort_values("weight", ascending=False).head(25))

## Save extraction outputs

In [ ]:
save_table(entities_df, PATHS.data_processed / "notebook_entities.csv")
save_table(keywords_df, PATHS.data_processed / "notebook_keywords.csv")
save_table(relations_df, PATHS.data_processed / "notebook_relations.csv")
save_table(aggregated_df, PATHS.data_processed / "notebook_aggregated_relations.csv")

print("Saved extraction outputs to data/processed.")